<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Juba_Floods_2018_2020_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xee
import xee
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
import ee

In [ ]:
!pip install -U geemap

In [ ]:
import geemap

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
ssd = (ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filterBounds(roi)
)

map.addLayer(ssd, {}, "ssd")

In [ ]:
pr = (
    ee.ImageCollection("ECMWF/ERA5/MONTHLY")
    .filterDate('2018','2020')
    .select('total_precipitation')
    .map(lambda img: img.clip(ssd).copyProperties(img, ['system:time_start']))
)
pr

In [ ]:
ds = xr.open_dataset(
    pr,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.27,
    geometry = ssd.geometry()
)

In [ ]:
ds = ds.sortby('time') * 1000.0

In [ ]:
import matplotlib.pyplot as plt

plot = ds.total_precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 6,
    robust = True,
    cmap = 'Blues',
    levels = 15,
    add_colorbar = False # Disable automatic colorbar
)

# Get the figure object from the xarray FacetGrid
fig = plot.fig

# Create a new axes for the colorbar on the right side of the figure
# The values are [left, bottom, width, height] as a fraction of the figure width and height
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) # Adjust these values for precise positioning

# Create the colorbar using the first subplot's mappable for scale
cbar = fig.colorbar(plot.axes.flat[0].collections[0], cax=cbar_ax, orientation='vertical')
cbar.set_label('Precipitation [mm]', fontsize = 15)
cbar.ax.tick_params(labelsize = 15)

for ax in plot.axes.flat:
  ax.set_title(ax.get_title(), fontsize = 15)
  ax.set_xlabel(ax.get_xlabel(), fontsize = 15)
  ax.set_ylabel(ax.get_ylabel(), fontsize = 15)
  ax.tick_params(axis = 'both', labelsize = 15)
  ax.set_aspect('equal')

plt.suptitle('Annual Precipitation Contour Plot', y=1.02)

plt.tight_layout(rect=[0.02, 0.05, 0.9, 0.95])
plt.show()

plt.savefig('pr_ssd.png', bbox_inches = 'tight', dpi = 360)

In [ ]:
map2 = geemap.Map()
map2

In [ ]:
roi = map2.draw_last_feature.geometry()
roi

In [ ]:
sen1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterDate('2018', '2020')
    .filterBounds(roi)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('instrumentMode','IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
   )
sen1

In [ ]:
!pip install matplotlib
import matplotlib.pyplot as plt

In [ ]:
ds2 = xr.open_dataset(
    sen1,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.0009,
    geometry = roi
)
ds2

In [ ]:
ds2 = ds2.sortby('time') * 1

In [ ]:
ds2_monthly = ds2.resample(time = 'M').min('time')

In [ ]:
before = ds2_monthly.sel(time = '2019-03').isel(time = 0)
after = ds2_monthly.sel(time = '2019-04').isel(time = 0)
flood = before - after

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (12,3))
plt.tight_layout(w_pad = 4)

plot1 = before.VV.plot(
    x = 'lon',
    y = 'lat',
    robust = True,
    cmap = 'gray',
    ax = ax[0]
)

plot2 = after.VV.plot(
    x = 'lon',
    y = 'lat',
    robust = True,
    cmap = 'gray',
    ax = ax[1]
)

plot3 = flood.VV.plot(
    x = 'lon',
    y = 'lat',
    robust = True,
    cmap = 'gray',
    ax = ax[2],
    vmin = 0
)
plt.savefig('juba_flood.png', dpi = 360, bbox_inches = 'tight')